In [ ]:
import random
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
!pip install catboost
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")


SEED = 42
random.seed(SEED)
np.random.seed(SEED)


data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3),n_estimators=300,learning_rate=0.01,random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300,learning_rate=0.01,max_depth=3,subsample=0.8,random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    eval_metric="logloss", random_state=SEED, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, verbose=-1, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4,random_seed=SEED, verbose=0),



}

In [ ]:
rows = []
for name, model in models.items():
    t0 = time.time()
    cvr = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "roc_auc"], n_jobs=1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "Model": name,
        "CV Acc": cvr["test_accuracy"].mean(),
        "CV Acc Std": cvr["test_accuracy"].std(),
        "CV AUC": cvr["test_roc_auc"].mean(),
        "Test Acc": accuracy_score(y_test, pred),
        "Test AUC": roc_auc_score(y_test, proba),
        "Süre (s)": time.time() - t0,
    })
    print(f"{name:<22} tamamlandı")

results = pd.DataFrame(rows).sort_values("CV AUC", ascending=False).reset_index(drop=True)
pd.set_option("display.float_format", "{:.4f}".format)
print("\n", results.to_string(index=False))


Random Forest          tamamlandı
AdaBoost               tamamlandı
Gradient Boosting      tamamlandı
HistGradientBoosting   tamamlandı
XGBoost                tamamlandı
LightGBM               tamamlandı
CatBoost               tamamlandı

                Model  CV Acc  CV Acc Std  CV AUC  Test Acc  Test AUC  Süre (s)
            LightGBM  0.9701      0.0163  0.9948    0.9561    0.9917    1.3034
            CatBoost  0.9578      0.0128  0.9943    0.9561    0.9950    7.7169
             XGBoost  0.9649      0.0175  0.9939    0.9561    0.9944    1.7574
HistGradientBoosting  0.9613      0.0233  0.9917    0.9737    0.9937    4.4458
   Gradient Boosting  0.9526      0.0131  0.9909    0.9298    0.9904   19.6767
       Random Forest  0.9543      0.0102  0.9896    0.9561    0.9931    5.1536
            AdaBoost  0.9578      0.0130  0.9879    0.9211    0.9864   18.8131
